current_price (O preço listado no anúncio).
market_median_price (O preço de mercado daquele exato modelo/configuração).
potential_profit (Lucro bruto projetado em R$).
profit_margin_pct (Margem em porcentagem).
opportunity_score (Um ranking interno de 0 a 100 baseado em margem, urgência e condição).

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, insert, func, cast, String
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.config import db_engine
from app.models import (
    SilverCleanAd,
    GoldPriceVariationAlert,
    GoldMarketBaseline,
    GoldMarketTrend
)

In [ ]:
with db_engine.connect() as connection:
    # Raw data
    df_silver_raw = pl.read_database(
        select(
            SilverCleanAd.ad_id,
            SilverCleanAd.title,
            SilverCleanAd.url,
            SilverCleanAd.first_image_src,
            SilverCleanAd.item_condition,
            SilverCleanAd.price.label('current_price'),
            SilverCleanAd.baseline_id,
            func.concat_ws(
                ', ', 
                func.nullif(SilverCleanAd.cpu_model, 'Not Informed'),
                cast(func.nullif(SilverCleanAd.ram_gb, 0), String) + 'GB RAM',
                cast(func.nullif(SilverCleanAd.storage_gb, 0), String) + 'GB SSD'
            ).label('specs_summary'),
            SilverCleanAd.urgent_sale.label('is_urgent_sale')
        ),
        connection=connection
    )

    # Baselines
    df_gold_baselines = pl.read_database(
        select(
            GoldMarketBaseline.baseline_id,
            GoldMarketBaseline.median_price.label('market_median_price')
        ),
        connection=connection
    )

In [ ]:
if not df_gold_market_trends.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(GoldPriceVariationAlert), df_gold_market_trends.to_dicts()
        )
else:
    print("No data found to insert.")